This notebook uses the following inputs:
1. Harmonised socioeconomic indicators for the base year, saved as a geopackage file (output from step 1: **baseyear_prep**)
2. Projected national population counts from the SSP Scenario Explorer, saved as an Excel workbook
3. Projected national GDP from the SSP Scenario Explorer, saved as an Excel workbook
4. Projected regional Gini indices from Narayan et al., saved as an Excel workbook

in order to project socioeconomic indicators on a cell-by-cell basis, according to the following:
- Sub-national population distribution is assumed to remain the same across scenarios and transition years
- Sub-national income levels are calculated using a parameterised lognormal distribution, which depends on the rank of the cell in the base year, and the projected national mean income level and Gini index for each scenario and transition year

The resulting datasets are saved as shapefiles used later for electricity demand projection.

## 1. Importing required packages

The following cell needs to be run first whenever the kernel is restarted.

In [ ]:
from geocube.vector import vectorize

import geopandas as gpd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import numpy as np

import pandas as pd

import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.plot import show
from rasterio.features import rasterize
from rasterio.transform import from_origin

import rioxarray

from rtree import index

from scipy.optimize import curve_fit
from scipy.special import erf, erfinv
from scipy.stats import lognorm

from shapely.ops import nearest_points
from shapely.geometry import box

from sklearn.metrics import r2_score

import mapclassify

## 2. Country list and input files

In [ ]:
# User-defined country code and country name dictionary
ccode_dict = {'NAM': 'Namibia'}

# SSP scenarios and transition period
n_scenario = 5 # number of SSPs
start_year = 2025
end_year = 2050
year_int = 5 # length of each time step between the start and end years

In [ ]:
# Import global socioeconomic projections, according to the 5 SSPs
for n in range(1, n_scenario + 1):
    name_pop = 'SSP' + str(n) + '_POP_All'
    name_gdp = 'SSP' + str(n) + '_GDP_All'
    
    # importing GDP and POP files for each SSP (2025-2100, all countries)
    locals()[name_pop] = pd.read_excel('input/SSP_POP/SSP' + str(n) + '_POP_2025_2100_AllCountries.xlsx')
    locals()[name_gdp] = pd.read_excel('input/SSP_GDP/SSP' + str(n) + '_GDP_2025_2100_AllCountries.xlsx')

# importing Gini index projections (all SSPs, years, and GCAM regions)
Gini_GCAM = pd.read_excel('input/GCAM_Gini_SSP_1967_2100.xlsx', sheet_name='Gini_regions_filtered')
# importing GCAM region IDs (to map countries to GCAM regions)
GCAM_regID = pd.read_excel('input/GCAM_Gini_SSP_1967_2100.xlsx', sheet_name='iso_GCAM_regID')
# importing base year Gini indices (GCAM regions)
Gini_GCAM_base = pd.read_excel('input/GCAM_Gini_SSP_1967_2100.xlsx', sheet_name='Gini_regions_2015')
# importing base year Gini indices (all countries)
Gini_WB = pd.read_excel('input/WB_Gini_2015.xlsx', sheet_name='Gini_2015')

In [ ]:
# Import harmonised base year socioeconomic data for all countries under investigation
# (output of baseyear_prep)
for ccode in ccode_dict:
    base_name = ccode + '_base'
    sorted_name = ccode + '_sorted'
    
    locals()[base_name] = gpd.read_file('1_output/' + ccode + '_socioecon_base_2015_Kummu.gpkg')

    # calculate normalised population (i.e. pop share) for each cell
    locals()[base_name]['Norm_POP'] = locals()[base_name]['2015_POP'] / (locals()[base_name]['2015_POP'].sum() + 0.000001)

    # sort cells ascendingly according to their income level
    locals()[sorted_name] = locals()[base_name].sort_values(by=['2015_INC_corr'])
    
    # calculate the cumulative share of population
    locals()[sorted_name]['POP_cshare'] = locals()[sorted_name]['2015_POP'].cumsum() / (locals()[sorted_name]['2015_POP'].sum() + 0.000001)
    
    # calculate the cumulative share of GDP
    locals()[sorted_name]['GDP_cshare'] = locals()[sorted_name]['2015_GDP_corr'].cumsum() / locals()[sorted_name]['2015_GDP_corr'].sum()

# display one of the resulting geodataframes    
NAM_sorted.head()

## 3. Socioeconomic indicators projection

This part involves two main steps. The first step is retrieving population, total GDP, and Gini index projections for the study region (country) across different Shared Socioecnomic Pathway narratives. The second step is disaggregating these single-point projections over the study region, as follows:
- Total population is disaggregated using the normalised population of each cell from the base year.
- Total GDP is disaggregated using a lognormal income distribution, which is parametrised via mean income and Gini index.

Finally, the disaggregated socio-economic projections are rasterised and saved as high-resolution maps.

### 3.1 Retrieving national-level population and GDP projections

In [ ]:
for ccode in ccode_dict:
    pop_proj_name = ccode + '_pop_proj'
    gdp_proj_name = ccode + '_gdp_proj'
    inc_proj_name = ccode + '_inc_proj'
    
    # initialising population, GDP, and mean income (i.e. GDP per capita) dataframes
    locals()[pop_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})
    locals()[gdp_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})
    locals()[inc_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})

    # populating the constructed dataframes
    for n in range(1, n_scenario + 1):
        name_pop = 'SSP' + str(n) + '_POP_All'
        name_gdp = 'SSP' + str(n) + '_GDP_All'
        
        # creating initial zero columns for each SSP
        col_name = 'SSP' + str(n)
        locals()[pop_proj_name][col_name] = 0.0
        locals()[gdp_proj_name][col_name] = 0.0
        
        for x in range(start_year, end_year + year_int, year_int):
            # original SSP datasets have population count in millions
            locals()[pop_proj_name].loc[locals()[pop_proj_name]['Year'] == x, [col_name]] = (locals()[name_pop].loc[locals()[name_pop]['Region'] == ccode_dict[ccode], str(x)].values[0] * 10 ** 6).round(0)
            # original SSP datasets have total GDP in billions
            locals()[gdp_proj_name].loc[locals()[gdp_proj_name]['Year'] == x, [col_name]] = locals()[name_gdp].loc[locals()[name_gdp]['Region'] == ccode_dict[ccode], str(x)].values[0] * 10 ** 9
            # calculating national average income level based on retrieved population and GDP projections
            locals()[inc_proj_name].loc[locals()[inc_proj_name]['Year'] == x, [col_name]] = \
            locals()[gdp_proj_name].loc[locals()[gdp_proj_name]['Year'] == x, [col_name]] / locals()[pop_proj_name].loc[locals()[pop_proj_name]['Year'] == x, [col_name]]

# displaying population dataframe as an example
NAM_pop_proj

### 3.2 Retrieving regional inequality projections

Inequality projections, i.e. Gini index projections, are available from Narayan et al. on a regional level (GCAM regions). To obtain the Gini index projections for the country under investigation, the following steps are used:
1. Map country name to GCAM region ID
2. Use GCAM region ID to retrieve the corresponding Gini index projections for each year and SSP narrative
3. Adjust Gini index projections to the starting point of the country in question, i.e. the same development trand is followed, but the starting Gini index is different

In [ ]:
for ccode in ccode_dict:
    gini_proj_name = ccode + '_gini_proj'
    gcam_id_name = ccode + '_gcam_id'
    
    # initialising gini index dataframes
    locals()[gini_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})

    # retrieving corresponding GCAM region ID
    locals()[gcam_id_name] = GCAM_regID.loc[GCAM_regID['iso'] == ccode.lower(), 'GCAM_region_ID'].values[0]

    # retrieving base year Gini indices for country and corresponding GCAM region
    cgini_base = Gini_WB.loc[Gini_WB['Country Code'] == ccode, 'Gini_2015'].values[0] # country
    rgini_base = Gini_GCAM_base.loc[Gini_GCAM_base['GCAM_region_ID'] == locals()[gcam_id_name], 'gini'].values[0] # GCAM region
    corr_fac = cgini_base / rgini_base # correction factor to be applied to regional Gini projections

    # populating the constructed dataframe
    for n in range(1, n_scenario + 1):        
        # creating initial zero columns for each SSP
        col_name = 'SSP' + str(n)
        locals()[gini_proj_name][col_name] = 0.0
        
        for x in range(start_year, end_year + year_int, year_int):
            # retrieving pojected Gini index for each year and SSP narrative,
            # then multiplying by correction factor to align GCAM trend with country starting point
            locals()[gini_proj_name].loc[locals()[gini_proj_name]['Year'] == x, [col_name]] = (Gini_GCAM.loc[(Gini_GCAM['GCAM_region_ID'] == locals()[gcam_id_name]) & (Gini_GCAM['year'] == x) & (Gini_GCAM['sce'] == col_name), 'gini'].values[0]) * corr_fac

# displaying one of the resulting dataframes
NAM_gini_proj

### 3.3 Disaggregating national-level population projections

In [ ]:
# The following disaggregation assumes that historical population distribution remains the same
for ccode in ccode_dict:
    sorted_name = ccode + '_sorted'
    pop_proj_name = ccode + '_pop_proj'
    
    for n in range(1, n_scenario + 1):
        # creating one GeoDataFrame per country and SSP, including all modelled years
        socioecon_proj_name = ccode + '_socioecon_SSP' + str(n)
        locals()[socioecon_proj_name] = locals()[sorted_name].copy()

        for x in range(start_year, end_year + year_int, year_int):
            # calculating population count per SSP, year, and cell based on base year normalised population shares
            locals()[socioecon_proj_name][str(x)+'_POP'] = locals()[socioecon_proj_name]['Norm_POP'] * locals()[pop_proj_name].loc[locals()[pop_proj_name]['Year'] == x, ['SSP'+str(n)]].values[0]
            
# displaying one of the resulting GeoDataFrames as an example
NAM_socioecon_SSP1.head()

### 3.3 Disaggregating national-level income projections

In [ ]:
### LOG-NORMAL INCOME DISTRIBUTION USING ACKLAM'S APPROXIMATION ###

# ------------------------ Normal CDF & inverse CDF ------------------------ #

def norm_cdf(x: np.ndarray) -> np.ndarray:
    """Standard normal CDF Φ(x) using erf."""
    x = np.asarray(x, dtype=float)
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))

def norm_ppf(p: np.ndarray) -> np.ndarray:
    """
    Inverse standard normal CDF (Acklam’s approximation).
    Accurate to ~1e-9 over (0,1).
    """
    p = np.asarray(p, dtype=float)
    if np.any((p <= 0) | (p >= 1)):
        raise ValueError("All probabilities must lie strictly between 0 and 1 (exclusive).")

    a = np.array([-3.969683028665376e+01,  2.209460984245205e+02,
                  -2.759285104469687e+02,  1.383577518672690e+02,
                  -3.066479806614716e+01,  2.506628277459239e+00])
    b = np.array([-5.447609879822406e+01,  1.615858368580409e+02,
                  -1.556989798598866e+02,  6.680131188771972e+01,
                  -1.328068155288572e+01])
    c = np.array([-7.784894002430293e-03, -3.223964580411365e-01,
                  -2.400758277161838e+00, -2.549732539343734e+00,
                   4.374664141464968e+00,  2.938163982698783e+00])
    d = np.array([ 7.784695709041462e-03,  3.224671290700398e-01,
                   2.445134137142996e+00,  3.754408661907416e+00])

    plow, phigh = 0.02425, 1 - 0.02425
    x = np.empty_like(p)

    # Lower region
    mask = p < plow
    if np.any(mask):
        q = np.sqrt(-2 * np.log(p[mask]))
        x[mask] = (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                   ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)

    # Central region
    mask = (p >= plow) & (p <= phigh)
    if np.any(mask):
        q = p[mask] - 0.5
        r = q*q
        x[mask] = (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5]) * q / \
                   (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)

    # Upper region
    mask = p > phigh
    if np.any(mask):
        q = np.sqrt(-2 * np.log(1 - p[mask]))
        x[mask] = -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                    ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    return x

# ------------------------ Lognormal parameter mapping ------------------------ #

def lognormal_params_from_mean_gini(mean_income: float, gini_index: float) -> tuple[float, float]:
    """
    Map arithmetic mean and Gini to (mu, sigma) for a Lognormal.
      mean = exp(mu + 0.5*sigma^2)
      Gini = 2*Phi(sigma/sqrt(2)) - 1
    """
    if mean_income <= 0:
        raise ValueError("mean_income must be positive.")
    if not (0 < gini_index < 1):
        raise ValueError("gini_index must be in (0,1).")

    p = (gini_index + 1.0) / 2.0
    sigma = np.sqrt(2.0) * norm_ppf(np.array([p]))[0]
    mu = np.log(mean_income) - 0.5 * sigma**2
    return mu, sigma

# ------------------------ Quantiles and bin means from cumulative shares ------------------------ #

def incomes_at_cum_shares(mean_income: float,
                          gini_index: float,
                          cum_shares: np.ndarray,
                          *,
                          mode: str = "quantile") -> np.ndarray:
    """
    Return income levels corresponding to cumulative population shares.

    Parameters
    ----------
    mean_income : float
        Target arithmetic mean of the distribution.
    gini_index : float
        Target Gini coefficient (0 < G < 1).
    cum_shares : array_like
        Monotonic increasing cumulative population shares in (0,1]; e.g., [0.1, 0.2, ..., 1.0].
    mode : {'quantile', 'bin_mean'}
        'quantile' : returns Q(p_i) = exp(mu + sigma * Phi^{-1}(p_i)) at each cumulative share p_i.
        'bin_mean' : returns average income within each interval (p_{i-1}, p_i]; length equals cum_shares.size.

    Returns
    -------
    incomes : np.ndarray
        Income at each cumulative share (quantile), or average income within each cumulative bin (bin_mean).

    Notes
    -----
    - 'quantile' gives income thresholds; useful for Lorenz curves and percentiles.
    - 'bin_mean' gives income **averages per group**; if you weight each bin by its share width and multiply by population,
      total GDP equals mean_income × population_size (up to floating error).
    """
    cum_shares = np.asarray(cum_shares, dtype=float)
    if np.any((cum_shares <= 0) | (cum_shares > 1)) or np.any(np.diff(cum_shares) <= 0):
        raise ValueError("cum_shares must be strictly increasing and lie in (0,1], e.g., [0.1, 0.2, ..., 1.0].")

    mu, sigma = lognormal_params_from_mean_gini(mean_income, gini_index)

    if mode == "quantile":
        # Avoid endpoints that cause +/-inf in the inverse CDF
        eps = 1e-12
        p = np.clip(cum_shares, eps, 1 - eps)
        z = norm_ppf(p)
        incomes = np.exp(mu + sigma * z)
        return incomes

    elif mode == "bin_mean":
        # Compute mean income within each bin (p_{i-1}, p_i], using truncated lognormal moments.
        # Let Y ~ N(mu, sigma^2), X = exp(Y). For bounds a,b in Y-space:
        #   E[X | a<Y<b] = exp(mu + 0.5*sigma^2) *
        #                  [Phi((b - mu - sigma^2)/sigma) - Phi((a - mu - sigma^2)/sigma)] /
        #                  [Phi((b - mu)/sigma)           - Phi((a - mu)/sigma)]
        p_edges = np.concatenate(([0.0], cum_shares))           # include 0 as left edge
        # Map edges in probability space to Y-space bounds (allow +/-inf at 0 and 1)
        y_edges = np.empty_like(p_edges)
        mask0 = p_edges == 0.0
        mask1 = p_edges == 1.0
        maskm = (~mask0) & (~mask1)
        y_edges[mask0] = -np.inf
        y_edges[mask1] = +np.inf
        if np.any(maskm):
            y_edges[maskm] = mu + sigma * norm_ppf(p_edges[maskm])

        yL = y_edges[:-1]
        yU = y_edges[1:]

        # Denominator P(a<Y<b)
        denom = norm_cdf((yU - mu) / sigma) - norm_cdf((yL - mu) / sigma)
        # Numerator piece for E[exp(Y) 1{a<Y<b}]
        numer = norm_cdf((yU - mu - sigma**2) / sigma) - norm_cdf((yL - mu - sigma**2) / sigma)

        # Handle any tiny denominators robustly
        small = denom < 1e-15
        with np.errstate(divide='ignore', invalid='ignore'):
            bin_means = np.exp(mu + 0.5 * sigma**2) * (numer / denom)
        # If a bin is extremely small, fallback to midpoint approximation
        if np.any(small):
            # Use the quantile at bin mid-point
            p_mid = 0.5 * (p_edges[:-1] + p_edges[1:])
            z_mid = norm_ppf(np.clip(p_mid, 1e-12, 1-1e-12))
            bin_means[small] = np.exp(mu + sigma * z_mid[small])

        return bin_means

    else:
        raise ValueError("mode must be 'quantile' or 'bin_mean'.")


In [ ]:
# The following disaggregation utilises a parametrised lognormal income distribution based on projected mean income and Gini index
for ccode in ccode_dict:
    sorted_name = ccode + '_sorted'
    gdp_proj_name = ccode + '_gdp_proj'
    inc_proj_name = ccode + '_inc_proj'
    gini_proj_name = ccode + '_gini_proj'

    for n in range(1, n_scenario + 1):
        socioecon_proj_name = ccode + '_socioecon_SSP' + str(n)

        for x in range(start_year, end_year + year_int, year_int):
            # retrieving gini and mean income, used to parameterise the income distribution
            inc = locals()[inc_proj_name].loc[locals()[inc_proj_name]['Year'] == x, ['SSP'+str(n)]].values[0]
            gini = locals()[gini_proj_name].loc[locals()[gini_proj_name]['Year'] == x, ['SSP'+str(n)]].values[0]
            gdp = locals()[gdp_proj_name].loc[locals()[gdp_proj_name]['Year'] == x, ['SSP'+str(n)]].values[0] # for validation
            
            # distribute GDP using acklam's approximation
            cum_shares = locals()[socioecon_proj_name]['POP_cshare']
            
            locals()[socioecon_proj_name][str(x)+'_INC'] = incomes_at_cum_shares(inc, gini, cum_shares, mode="bin_mean")
            locals()[socioecon_proj_name][str(x)+'_GDP'] = locals()[socioecon_proj_name][str(x)+'_INC'] * locals()[socioecon_proj_name][str(x)+'_POP']
            
            # verify resulting total GDP
            print('Results for', ccode_dict[ccode], 'in', str(x), 'and SSP'+str(n))
            #print('gini =', locals()[gini_name])
            print('total GDP, after distribution =', f'{locals()[socioecon_proj_name][str(x)+'_GDP'].sum():,.0f}')
            print('total GDP, original =', f'{float(gdp.item()):,.0f}')
            print('relative error =', f'{(locals()[socioecon_proj_name][str(x)+"_GDP"].sum() - float(gdp.item())) / float(gdp.item()):,.2%}')
    

### 3.4 Saving socioeconomic projections to shapefiles

This step saves the projected population, GDP, and income data for all scenarios and transition years. National-level projections were based on SSP narratives for population count and total GDP, and Narayan et al. for SSP-consistent GCAM regional Gini indices.

- Sub-national disaggregation of population assumed the same historical distribution persists in the future.
- Sub-national disaggregation of GDP was done using a lognormal distribution parametrised via national-level mean income and Gini index.

High-resolution socioeconomic projections are used in another script to estimate residential electricity demand per populated cell, modelled year, and SSP narrative.

In [ ]:
for ccode in ccode_dict:
    for n in range(1, n_scenario + 1):
        socioecon_proj_name = ccode + '_socioecon_SSP' + str(n)

        locals()[socioecon_proj_name].set_geometry('geometry', inplace=True, crs='EPSG:4326')
        locals()[socioecon_proj_name].to_file('3_output/' + ccode + '_socioecon_SSP' + str(n) + '.gpkg', driver='GPKG')

## End of socioeconomic projection script